# Aula 02 — Titanic com Pipeline e ColumnTransformer

## Objetivo
Construir um modelo de classificação para prever sobrevivência no dataset Titanic, utilizando Pipeline + ColumnTransformer, com um baseline em Regressão Logística e uma melhoria simples para comparação.

##### Imports

In [2]:
import pandas as pd
import seaborn as sns
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

##### Carregar datasets

In [3]:
df_t = sns.load_dataset("titanic")

df_t = df_t.drop(columns=["alive"])

print("Primeiras linhas do dataset:")
display(df_t.head())

print("Shape do dataset:")
print(df_t.shape)

print("\nValores faltantes por coluna:")
print(df_t.isna().sum())

Primeiras linhas do dataset:


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,True


Shape do dataset:
(891, 14)

Valores faltantes por coluna:
survived         0
pclass           0
sex              0
age            177
sibsp            0
parch            0
fare             0
embarked         2
class            0
who              0
adult_male       0
deck           688
embark_town      2
alone            0
dtype: int64


##### Separar X e Y

In [4]:
y = df_t["survived"]
X = df_t.drop(columns=["survived"])

print("Shape de X:", X.shape)
print("Shape de y:", y.shape)

Shape de X: (891, 13)
Shape de y: (891,)


##### Split treino/teste

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (712, 13)
X_test: (179, 13)
y_train: (712,)
y_test: (179,)


##### Identificar colunas numéricas e categóricas  

In [6]:
num_cols = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_cols = X_train.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

print("Colunas numéricas:")
print(num_cols)

print("\nColunas categóricas:")
print(cat_cols)

Colunas numéricas:
['pclass', 'age', 'sibsp', 'parch', 'fare']

Colunas categóricas:
['sex', 'embarked', 'class', 'who', 'adult_male', 'deck', 'embark_town', 'alone']


##### Pré-processamento com Pipeline + ColumnTransformer

In [7]:
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer([
    ("num", num_pipeline, num_cols),
    ("cat", cat_pipeline, cat_cols)
])

##### Baseline

In [8]:
model_baseline_lr = Pipeline([
    ("prep", preprocess),
    ("clf", LogisticRegression(max_iter=1000, random_state=42))
])

model_baseline_lr.fit(X_train, y_train)
y_pred_baseline = model_baseline_lr.predict(X_test)

##### Avaliação do baseline

In [9]:
acc_baseline = accuracy_score(y_test, y_pred_baseline)
cm_baseline = confusion_matrix(y_test, y_pred_baseline)
tn_b, fp_b, fn_b, tp_b = cm_baseline.ravel()

print("=== BASELINE: Logistic Regression ===")
print(f"Accuracy: {acc_baseline:.4f}\n")

print("Classification Report:")
print(classification_report(y_test, y_pred_baseline))

print("Confusion Matrix:")
print(cm_baseline)

print(f"\nTN = {tn_b}")
print(f"FP = {fp_b}")
print(f"FN = {fn_b}")
print(f"TP = {tp_b}")

=== BASELINE: Logistic Regression ===
Accuracy: 0.8268

Classification Report:
              precision    recall  f1-score   support

           0       0.85      0.87      0.86       110
           1       0.79      0.75      0.77        69

    accuracy                           0.83       179
   macro avg       0.82      0.81      0.82       179
weighted avg       0.83      0.83      0.83       179

Confusion Matrix:
[[96 14]
 [17 52]]

TN = 96
FP = 14
FN = 17
TP = 52


##### Melhoria simples

In [10]:
model_melhoria = Pipeline([
    ("prep", preprocess),
    ("clf", LogisticRegression(
        max_iter=1000,
        random_state=42,
        class_weight="balanced"
    ))
])

model_melhoria.fit(X_train, y_train)
y_pred_melhoria = model_melhoria.predict(X_test)

##### Avaliação da melhoria

In [11]:
acc_melhoria = accuracy_score(y_test, y_pred_melhoria)
cm_melhoria = confusion_matrix(y_test, y_pred_melhoria)
tn_m, fp_m, fn_m, tp_m = cm_melhoria.ravel()

print("=== MELHORIA: Logistic Regression com class_weight='balanced' ===")
print(f"Accuracy: {acc_melhoria:.4f}\n")

print("Classification Report:")
print(classification_report(y_test, y_pred_melhoria))

print("Confusion Matrix:")
print(cm_melhoria)

print(f"\nTN = {tn_m}")
print(f"FP = {fp_m}")
print(f"FN = {fn_m}")
print(f"TP = {tp_m}")

=== MELHORIA: Logistic Regression com class_weight='balanced' ===
Accuracy: 0.8268

Classification Report:
              precision    recall  f1-score   support

           0       0.86      0.85      0.86       110
           1       0.77      0.78      0.78        69

    accuracy                           0.83       179
   macro avg       0.82      0.82      0.82       179
weighted avg       0.83      0.83      0.83       179

Confusion Matrix:
[[94 16]
 [15 54]]

TN = 94
FP = 16
FN = 15
TP = 54


##### Comparação objetiva

In [ ]:
comparacao = pd.DataFrame({
    "Modelo": ["Baseline LR", "LR balanceada"],
    "Accuracy": [acc_baseline, acc_melhoria],
    "TN": [tn_b, tn_m],
    "FP": [fp_b, fp_m],
    "FN": [fn_b, fn_m],
    "TP": [tp_b, tp_m]
})

display(comparacao.style.format({"Accuracy": "{:.4f}"}))

## Interpretação dos resultados

No baseline, o modelo apresentou mais erros na classe **[0 ou 1]**, o que pode ser observado pelos valores de **[FP/FN]** na matriz de confusão e também pelo recall da classe no classification report.  
Comparando os dois modelos, a melhoria fez com que os **FN [aumentassem/diminuíssem]** e os **FP [aumentassem/diminuíssem]**, mostrando um trade-off entre identificar mais sobreviventes e errar mais previsões positivas.  
Isso é comum quando usamos `class_weight="balanced"`, pois o modelo passa a dar mais atenção à classe minoritária ou ao equilíbrio entre as classes.  
Apesar da mudança, a accuracy sozinha não é suficiente para avaliar o modelo, porque ela não mostra claramente o custo dos erros por classe.  
Uma limitação do dataset é a presença de valores faltantes em várias colunas, além do viés histórico dos dados, já que o contexto do Titanic não representa todos os cenários reais de sobrevivência.  
Outra limitação é que variáveis importantes podem estar ausentes, o que reduz a capacidade do modelo de explicar completamente o fenômeno.